# Polynomials, Lagrange Interpolation and the Vandermonde Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial

## Task 1: Working with polynomials

$f(x) = -1 + 4x - 0.5x^2$

In [ ]:
# (a) build the polynomial from its coefficients (lowest power first)
p = Polynomial([-1.0, 4.0, -0.5])
print(p)

# (b) degree
print("Degree of the polynomial is", p.degree())

In [ ]:
# (c) plot f'(x) on [-1.5, 1.5]
p_prime = p.deriv(1)
x = np.linspace(-1.5, 1.5, 100)

plt.plot(x, p_prime(x))
plt.axhline(0, color='k', linewidth=0.8)
plt.title("f'(x)")
plt.xlabel("x")
plt.show()

In [ ]:
# (d) f'(x) / f''(x) at x = 2
p_dprime = p_prime.deriv(1)
x0 = 2
print(f"f'({x0}) / f''({x0}) =", p_prime(x0) / p_dprime(x0))

## Task 2: Lagrange interpolation

For nodes $(x_0, y_0), \dots, (x_n, y_n)$:

$$P(x) = \sum_{k=0}^{n} y_k \, l_k(x), \qquad l_k(x) = \prod_{j \ne k} \frac{x - x_j}{x_k - x_j}$$

In [ ]:
def l_k(data_x, k, x):
    """k-th Lagrange basis polynomial evaluated at x."""
    lag = 1.0
    for j in range(len(data_x)):
        if j != k:
            lag *= (x - data_x[j]) / (data_x[k] - data_x[j])
    return lag

def lagrange_interpolation(data_x, data_y, x):
    """Value of the Lagrange interpolating polynomial at x (x can be an array)."""
    return sum(data_y[k] * l_k(data_x, k, x) for k in range(len(data_x)))

def get_poly_lagrange(data_x, data_y):
    """The interpolating polynomial as a numpy Polynomial object."""
    poly = Polynomial([0.0])
    for k in range(len(data_x)):
        basis = Polynomial([1.0])
        for j in range(len(data_x)):
            if j != k:
                basis *= Polynomial([-data_x[j], 1.0]) / (data_x[k] - data_x[j])
        poly += data_y[k] * basis
    return poly

In [ ]:
data_x = np.array([-3.0, -2.0, -1.0, 0.0, 1.0, 3.0, 4.0])
data_y = np.array([-60.0, -80.0, 6.0, 1.0, 45.0, 30.0, 16.0])

P = get_poly_lagrange(data_x, data_y)
print(P)

# the polynomial passes through every node
assert np.allclose(lagrange_interpolation(data_x, data_y, data_x), data_y)
assert np.allclose(P(data_x), data_y)

x = np.linspace(-3.5, 4.5, 200)
plt.plot(x, lagrange_interpolation(data_x, data_y, x), label="Lagrange polynomial")
plt.scatter(data_x, data_y, color="red", zorder=3, label="Data points")
plt.legend()
plt.title("Lagrange interpolation")
plt.show()

## Task 3: Vandermonde matrix

Interpolation as a linear system $V a = y$, where $V_{ij} = x_i^{\,j}$ and $a$ holds the polynomial coefficients.

In [ ]:
def vandermonde(x):
    """Vandermonde matrix with V[i][j] = x[i]**j."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    V = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            V[i][j] = x[i] ** j
    return V

V = vandermonde([5, 6, 7])
print(V)
assert np.allclose(V, np.vander([5, 6, 7], increasing=True))

In [ ]:
# solve for the coefficients and check against the Lagrange result
coeffs = np.linalg.solve(vandermonde(data_x), data_y)
print("Vandermonde coefficients:", np.round(coeffs, 6))
print("Same as Lagrange:", np.allclose(coeffs, P.coef))